# DELTA LAKE STORAGE

## 1. Delta Lake

#### What is Delta Lake?
* Delta Lake is an open-source storage layer that brings ACID transaction, scalable metadata handling, data versioning, time travel capabilities and unified support for streaming/batch processing to data lakes.
* It is default storage layer in Databricks.
* Delta Lake runs on top of existing Data Lakes (cloud storage like AWS S3, ADLF, etc.).
<br>
##### Features:
* ACID transaction
* Schema Enforcement
* Scalable Metadata handling
* DML Operations
* Data Versoning/ Time Travel
* Transaction Log
* Streaming/batch data processing

## 2. Understanding Traditional Data Storage Capabilities

### Data Warehouses

* Technical: Cleaned + Structured data
* Business: Operational data/ Analytical data
* Storage: Managed warehouse storage
* ACID transactions
* High Cost
* Schema-on-write
* Strong Data Modeling
* Analytical Processing/ Fast DB queries for Reporting
* Cons:
    * Less flexible
    * Expensive storage & compute
    * Data Preprocessing required
* Use Case: BI Reports/Dashboards


![image_1793511941367.png](./image_1793511941367.png "image_1793511941367.png")

### Data Lake
* Technical: Wide varity of data  
* Business: Central repository for all enterprise data
* Storage: Object Storage
* Flexible
* Low storage cost
* Schema-on-read
* Raw data is preserved (uncleaned)
* Usually built on object storage
* Very large scale  
* Use Case: Big Data, AI/ML, IoT, Analytics
* Cons:
    * Data Swamp
    * Lack of ACID Transactions

![image_1793511987586.png](./image_1793511987586.png "image_1793511987586.png")

## 3. Delta Table Architecture

* Delta Table is essentially Data files + Transactional Log + Metadata
* When a Delta Table is created, a new directory gets created in the specified storage location (usually in ADLS Gen2, S3, or local DBFS). This directory contains:
    * _delta_log (directory): Stores the metadata + transaction logs.
    * .parquet (files): The actual data stored in columnar format.
* With each commit a delta log file is created along with its corresponding parquet file. 

![image_1793520855468.png](./image_1793520855468.png "image_1793520855468.png")

### _delta_log

* all the transactional log files are stored in this _delta_log directory
* it tells Delta which Parquet files constitute the table at each version.
* these transacttional logs contains the metadata + transaction history
* this is what makes ACID + Time Travel possible
* this directory contains:
    * .json files: Every change to the table is recorded as a JSON file.
    * .checkpoint.parquet files: A compacted representation of the Delta transaction-log state.

#### .json log files



* _**Each json represents an action**_
* Delta json log file contains the actions and information associated with a particular table commit
* Json/metadata file contains info. related to actions and associated metadata for a particular commit, corresponsing parquet files, parititons and etc.
* Since the transaction log preserves the history of each commit, it makes Time Travel possible and helps optimize reads by allowing Delta to efficiently identify the relevant data files that need to be read.
* Whenever commit happens it creates a json file inside _delta_log directory that contains information related to that partuclar commit and corresponding parquet file. 
* _json log_ : _parquet_: 1:N  
* These actions describe what changed in that commit, such as:- 
    * add → a Parquet file was added to the logical table
    * remove → a Parquet file was removed from the logical table
    * metaData → table schema, partition columns, table properties, etc.
    * protocol → Delta reader/writer protocol requirements
    * commitInfo → information about the operation/commit
    * txn → transaction/application information when applicable

In [0]:
%sql
    SELECT commitInfo 
    FROM json.`/Volumes/workspace/default/demo/simple_table/_delta_log/00000000000000000000.json`
    WHERE commitInfo IS NOT NULL;

```
"commitInfo": { 
    "timestamp": 1587968586154,
    "operation": "WRITE",
    "operationParameters": {
      "mode": "ErrorIfExists",
      "partitionBy": []
    },
    "isBlindAppend": true
  }
  ```  

In [0]:
%sql
SELECT metaData
FROM json.`/Volumes/workspace/default/demo/simple_table/_delta_log/00000000000000000000.json`
WHERE "metaData IS NOT NULL;


```
  "metaData": {
    "id": "5fba94ed-9794-4965-ba6e-6ee3c0d22af9",
    "format": {
      "provider": "parquet",
      "options": {}
    },
    "schemaString": "{\"type\":\"struct\",\"fields\":[{\"name\":\"id\",\"type\":\"long\",\"nullable\":true,\"metadata\":{}}]}",
    "partitionColumns": [],
    "configuration": {},
    "createdTime": 1587968585495
  ```

In [0]:
%sql
SELECT add
FROM json.`/Volumes/workspace/default/demo/simple_table/_delta_log/00000000000000000000.json`
WHERE add IS NOT NULL LIMIT 1;

```
{
      "path": "part-00000-a72b1fb3-f2df-41fe-a8f0-e65b746382dd-c000.snappy.parquet",
      "partitionValues": {},
      "size": 262,
      "modificationTime": 1587968586000,
      "dataChange": true
    }
```

#### .checkpoint.parquet

* **Snapshot of the Delta table's state at a particular version**
* After every n commits, a checkpoint .parquet is written to avoid reading thousands of JSON files.
* Over time, a Delta table can accumulate many JSON transaction-log files. Reading and replaying a large number of JSON files to reconstruct the current table state can become increasingly expensive. Therefore, Delta periodically creates a checkpoint, which stores the accumulated table state in a compact Parquet file. This allows Delta to start from the checkpoint and process only the JSON log files created after that checkpoint, reducing the number of files and I/O operations required to reconstruct the table state.
* A checkpoint action accumulates and compacts the actions from the Delta JSON logs, storing only the resulting state needed to represent a particular table version.

```text
JSON Logs
   ↓
ADD / REMOVE / UPDATE actions
   ↓
Accumulate + resolve actions
   ↓
Checkpoint
   ↓
Resulting state
   ↓
Table state at Version N
```

In [0]:
# Disable the safety check that forces format("delta")
spark.conf.set("spark.databricks.delta.formatCheck.enabled", "false")
# Force read the individual parquet file
df_spark = spark.read.format("parquet").load("/Volumes/workspace/default/demo/00000000000000000002.checkpoint.parquet")
display(df_spark)
# Reset the configuration back to true for safety
spark.conf.set("spark.databricks.delta.formatCheck.enabled", "true")


In [0]:
# Here we can see all the 'columns' that contains all the actions after being compacted
df_spark.columns
# ['txn', 'add', 'remove', 'metaData', 'protocol', 'domainMetadata']

In [0]:
df_spark.select('metaData').where(df_spark.metaData.isNotNull()).limit(1).toPandas()['metaData'].tolist()
#

Here is a snapshot from "add" column from the **checkpoint** file
```text
{'path': 'part-00000-1e14ba22-3114-46d1-96fb-48b4912507ce-c000.snappy.parquet',
  'partitionValues': {},
  'size': 5072,
  'modificationTime': 1723768498986,
  'dataChange': False,
  'tags': None,
  'deletionVector': None,
  'baseRowId': None,
  'defaultRowCommitVersion': None,
  'clusteringProvider': None,
  'stats': '{"numRecords":1,"minValues":{"id":1},"maxValues":{"id":1},"nullCount":{"id":0,"v":0,"array_of_variants":0,"struct_of_variants":{"v":0},"map_of_variants":0,"array_of_struct_of_variants":0,"struct_of_array_of_variants":{"v":0}}}'}
 ```

Here is a snapshot from "metaData" column from the **checkpoint** file
```
{'id': 'd7eb0848-b002-4e0b-9d8d-dd335c90946f',
  'name': None,
  'description': None,
  'format': {'provider': 'parquet', 'options': {}},
  'schemaString': '{"type":"struct","fields":[{"name":"id","type":"long","nullable":true,"metadata":{}},{"name":"v","type":"variant","nullable":true,"metadata":{}},{"name":"array_of_variants","type":{"type":"array","elementType":"variant","containsNull":true},"nullable":true,"metadata":{}},{"name":"struct_of_variants","type":{"type":"struct","fields":[{"name":"v","type":"variant","nullable":true,"metadata":{}}]},"nullable":true,"metadata":{}},{"name":"map_of_variants","type":{"type":"map","keyType":"string","valueType":"variant","valueContainsNull":true},"nullable":true,"metadata":{}},{"name":"array_of_struct_of_variants","type":{"type":"array","elementType":{"type":"struct","fields":[{"name":"v","type":"variant","nullable":true,"metadata":{}}]},"containsNull":true},"nullable":true,"metadata":{}},{"name":"struct_of_array_of_variants","type":{"type":"struct","fields":[{"name":"v","type":{"type":"array","elementType":"variant","containsNull":true},"nullable":true,"metadata":{}}]},"nullable":true,"metadata":{}}]}',
  'partitionColumns': array([], dtype=object),
  'configuration': {'delta.checkpointInterval': '2'},
  'createdTime': 1723768495302}
  ```

### .parquet Data file

* Here is where actual data is stored in parquet files
* **Delta never modified files - only adds new ones** i.e. Old files are NOT deleted on UPDATE/DELETE. New files are added; old ones are tombstoned in the log.
* Parquet is a column-oriented storage format designed for efficient querying through inherent optimization techniques like column pruning

In [0]:
# Disable the safety check that forces format("delta")
spark.conf.set("spark.databricks.delta.formatCheck.enabled", "false")
# Force read the individual parquet file
df_spark = spark.read.format("parquet").load("/Volumes/workspace/default/demo/part-00000-a496f40c-e091-413a-85f9-b1b69d4b3b4e-c000.snappy.parquet")
display(df_spark)
# Reset the configuration back to true for safety
spark.conf.set("spark.databricks.delta.formatCheck.enabled", "true")

**Output upon querying the data in the .parquet file.**
| date | county | state | fips | cases | deaths |
| :--- | :--- | :--- | :--- | :--- | :--- |
| 1/21/2020 | Snohomish | Washington | 53061 | 1 | 0 |
| 1/22/2020 | Snohomish | Washington | 53061 | 1 | 0 |
| 1/23/2020 | Snohomish | Washington | 53061 | 1 | 0 |
| 1/24/2020 | Cook | Illinois | 17031 | 1 | 0 |
| 1/24/2020 | Snohomish | Washington | 53061 | 1 | 0 |
| 1/25/2020 | Orange | California | 6059 | 1 | 0 |
| 1/25/2020 | Cook | Illinois | 17031 | 1 | 0 |
| 1/25/2020 | Snohomish | Washington | 53061 | 1 | 0 |
| 1/26/2020 | Maricopa | Arizona | 4013 | 1 | 0 |


### What happens in the background?

#### Creating a Delta Table:
```
CREATE TABLE employees (
    id INT,
    name STRING,
    department STRING,
    salary INT
) USING DELTA;
```
When the table is created:
* Databricks creates the Delta table structure/directory in storage - folder name same as table name.
* Directory will have only __delta_log_ directory.
* a JSON file named 00000000000000000000.json is written to __delta_log/_ that contains metadata/schema and properties about the table.
* Parquet data files = none


#### Inserting Data into a Delta Table:
```
INSERT INTO employees VALUES
(1, 'Alice', 'IT', 100000),
(2, 'Bob', 'HR', 80000),
(3, 'Charlie', 'IT', 90000);
```
When data is inserted into a Delta Table:
* Writes the data as parquet file/s  in the Delta Table directory: /mnt/delta/stock_data/part-00000-tid-12345.parquet
* Updates the transaction log with a new JSON entry: 00000000000000000001.json -> _delta_log/
* Supporse we execute another INSERT, Delta will create another parquet file with corresponding json log instead of modifying exisiting file.

#### Updating rows in a Delta Table
```
UPDATE employees
SET salary = 110000
WHERE id = 1;
```
When data is updated into a Delta Table:
* Identifies the existing Parquet file containing the record -> /mnt/delta/stock_data/part-00000-abc123.parquet/
* Reads the data, modifies the necessary rows, and writes a new Parquet file. -> /mnt/delta/stock_data/part-00002-new456.parquet/ & 00000000000000000002.json
* Marks the old file as removed in the transaction log.
```
{
  "remove": {
    "path": "part-00000-abc123.parquet"
  }
}
```
* Note: 
    * The old file is marked as removed but not physically deleted (this is handled during compaction).
    * It follows a copy-on-write (COW) mechanism.

#### Deleting rows in a Delta Table
```
DELETE FROM employees
WHERE id = 2;
```
When data is deleted into a Delta Table:
* Identifies the Parquet file containing the record.
* Writes a new Parquet file excluding the deleted row.
* Marks the old file as removed in the transaction log.

## 4. DELTA LAKE OPERATIONS

### 1. CREATE DELTA TABLE

* Defines the table schema and metadata.
* The table can be created as managed or external.
* An empty table generally has no business-data Parquet files yet.
* Can create a table from an existing Parquet dataset.
* Can also create a table using CTAS (CREATE TABLE AS SELECT).

**SparkSQL**
```
CREATE TABLE <table_name> (
    column_name1 data_type(),
    column_name2 data_type(),
    ....
)
USING DELTA;
```

**PySpark (DataFrame API)**
```
#1. Define the schema for your empty table
schema = StructType([
    StructField("<colume_name1>", DataType(), True),
    StructField("<colume_name2>", DataType(), True),
    ...
])

#2. Create an empty DataFrame using the schema
empty_df = spark.emptyDataFrame(schema)

#3. Save the empty DataFrame as a Delta table
empty_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("default.my_empty_delta_table") 
```

In [0]:
%sql
-- SparkSQL Syntax: To create delta table employees
CREATE TABLE employees (
    id INT,
    name STRING,
    department STRING,
    salary DECIMAL(10,2)
)
USING DELTA;

In [0]:
# PySpark Syntax: To create delta table employees
# from pyspark.sql.types import StructType, StructField, IntegerType, StringType

# 1. Define the schema for your empty table
schema = StructType([
    StructField("id", IntegerType(), True),
    StructField("rate", IntegerType(), True),
    StructField("quantity", IntegerType(), True)
])

# 2. Create an empty DataFrame using the schema
orders_df = spark.emptyDataFrame(schema)

# 3. Save the empty DataFrame as a Delta table
orders_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("default.orders")

In [0]:
from pyspark.sql import functions as F

orders_description = spark.sql("DESCRIBE EXTENDED orders").alias("orders")
employees_description = spark.sql("DESCRIBE EXTENDED employees").alias("employees")

result = (
    orders_description
    .join(
        employees_description,
        on="col_name",
        how="full_outer"
    )
    .select(
        F.col("col_name"),
        F.col("orders.data_type").alias("data_type_orders"),
        F.col("employees.data_type").alias("data_type_employees")
    )
)

display(result)

#### Code Variants for Specific Scenarios

##### Create using CTAS

In [0]:
%sql
CREATE TABLE employees_staging
USING DELTA
AS
SELECT *
FROM employees;

##### Create with partitioning

In [0]:
%sql
CREATE TABLE employees (
    id INT,
    name STRING,
    department STRING,
    salary DECIMAL(10,2),
    year INT
)
USING DELTA
PARTITIONED BY (year);

##### Create an external Delta table

In [0]:
%sql
CREATE TABLE employees(
    id INT,
    name STRING,
    department STRING,
    salary DECIMAL(10,2),
    year INT
)
USING DELTA
LOCATION '/mnt/data/employees';

Important Links:
* https://github.com/delta-io/delta-rs/tree/main/crates/test/tests/data/simple_table
* https://medium.com/@jithujosekokken/understanding-delta-tables-in-databricks-a-deep-dive-into-the-transaction-log-77c31467b99f
* https://pyspark.in/tutorials/spark/Delta%20Lake/868
* https://docs.databricks.com/aws/en/delta/tutorial?language=Python